# Prediction is not explanation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/data-analysis/diagnostic-predictive-analytics.ipynb)

**Worked Example** · UvA Business Analytics teaching collection

Work through the questions before running each cell. Explain the result, check its assumptions, and change one input to test your understanding.


## Setup

The libraries used below are already available in Colab; no installation cell is needed. For local Jupyter use, see the [environment guidance](https://github.com/gromicho/teaching/blob/main/docs/SETUP.md).


## Fit, test, interpret
We use the historical height–weight teaching dataset to illustrate regression. The original dataset provenance and population representativeness are not fully established; this is a computational example, not health advice or a normative model of bodies. Do not infer causality from an association.

We reserve test observations before fitting. A model that reproduces the training data need not predict new observations well.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib

# Prefer the checked-in file locally; Colab downloads the same frozen edition.
data_path = next((p for p in [Path("data/weight-height.csv"), Path("weight-height.csv")]
                 if p.is_file()), Path("weight-height.csv"))
if not data_path.is_file():
    url = "https://raw.githubusercontent.com/gromicho/teaching/main/data/weight-height.csv"
    with urlopen(url, timeout=45) as response:
        payload = response.read()
    if hashlib.sha256(payload).hexdigest() != "7c97d452e40c2a4242efb13a92303c6a6bbe5f4b5a0dd44fc4155a86836d8502":
        raise ValueError("Dataset checksum mismatch; do not use an unverified copy.")
    data_path.write_bytes(payload)
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == "7c97d452e40c2a4242efb13a92303c6a6bbe5f4b5a0dd44fc4155a86836d8502", "Unexpected local data version"


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
df = pd.read_csv(data_path)
print(df.columns.tolist())
df.head()


In [ ]:
X = df[['Height']]
y = df['Weight']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=42)
model = LinearRegression().fit(X_train,y_train)
predicted = model.predict(X_test)
print('Held-out MAE:', mean_absolute_error(y_test,predicted))
print('Held-out R-squared:', r2_score(y_test,predicted))
assert len(predicted) == len(y_test)
assert np.isfinite(predicted).all()


## Compare against a baseline
Would predicting the same training mean for everybody already perform well? The comparison is meaningful only on the same held-out observations.


In [ ]:
baseline = np.full(len(y_test), y_train.mean())
assert mean_absolute_error(y_test,predicted) < mean_absolute_error(y_test,baseline)
plt.scatter(X_test['Height'], y_test-predicted, s=4, alpha=0.3)
plt.axhline(0,color='black')
plt.xlabel('Height (original dataset units)')
plt.ylabel('Residual weight (original dataset units)')
plt.show()


## From prediction to judgment
What patterns remain in the residuals? Would this model transfer to children, another population or a different measurement protocol? Explain why a high R-squared does not settle any of these questions.

> **Optional UvA AI Chat prompt:** Ask me to distinguish training error, test error and causal explanation using this example. Ask for evidence for each claim and challenge any conclusion that goes beyond the data.
